# 04.02 · Embeddings & self-attention

**Chapter:** Transformers & How LLMs Work · **Milestone:** M1

## Goals
- [ ] Token embeddings + positional information
- [ ] Scaled dot-product attention from scratch (queries, keys, values)
- [ ] Causal masking: why a language model can't look ahead
- [ ] Multi-head attention, checked against PyTorch's built-in

## Resources
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) — free
- [3Blue1Brown – Attention in transformers, visually explained](https://www.youtube.com/watch?v=eMlx5fFNoYc) — free
- [Attention Is All You Need (paper)](https://arxiv.org/abs/1706.03762) — free

**Attention in one sentence**: each token builds a new representation of itself as a weighted average of all (previous) tokens,
where the weights come from how well its *query* matches each token's *key*.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

torch.manual_seed(0)
rng = np.random.default_rng(0)

## 1. Token embeddings
An embedding table is a `(vocab_size, d_model)` matrix; token ID `i` → row `i`. It's learned like any other weight.

In [ ]:
vocab_size, d_model = 50, 16
emb = nn.Embedding(vocab_size, d_model)
ids = torch.tensor([[3, 14, 15, 9]])  # (batch=1, seq=4)
print(emb(ids).shape)  # (1, 4, 16)
assert torch.equal(emb(ids)[0, 1], emb.weight[14])  # it's just a lookup

## 2. Position
Attention itself is order-blind (a weighted average doesn't care about order). Position must be injected:
- **Learned absolute** position embeddings (GPT-2): another table, indexed by position, added to token embeddings.
- **Sinusoidal** (original transformer): fixed sin/cos waves of different frequencies.
- **RoPE** (rotary, used by Llama/Qwen/Mistral/most modern LLMs): rotates query/key vectors by an angle proportional to position,
  so their dot product depends on *relative* distance.

In [ ]:
def sinusoidal(seq_len, d):
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d // 2)[None, :]
    angles = pos / 10000 ** (2 * i / d)
    return np.concatenate([np.sin(angles), np.cos(angles)], axis=1)

plt.imshow(sinusoidal(100, 64).T, aspect="auto", cmap="RdBu"); plt.xlabel("position"); plt.ylabel("dimension")
plt.title("sinusoidal position encodings"); plt.colorbar(); plt.show()

## 3. Scaled dot-product attention, from scratch
For input $X$ (seq × d): $Q = XW_Q$, $K = XW_K$, $V = XW_V$.

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- $QK^T$: every query dotted with every key → (seq × seq) score matrix.
- $\div\sqrt{d_k}$: keeps scores from growing with dimension (else softmax saturates).
- softmax per row → attention weights summing to 1.
- × $V$: weighted average of value vectors.

In [ ]:
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def attention_np(X, Wq, Wk, Wv, causal=True):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    scores = Q @ K.T / math.sqrt(K.shape[-1])
    if causal:
        mask = np.triu(np.ones_like(scores, dtype=bool), k=1)  # True above the diagonal = future tokens
        scores = np.where(mask, -np.inf, scores)
    weights = softmax(scores)
    return weights @ V, weights


seq, d, d_k = 6, 16, 8
X = rng.normal(size=(seq, d))
Wq, Wk, Wv = (rng.normal(size=(d, d_k)) / math.sqrt(d) for _ in range(3))
out, weights = attention_np(X, Wq, Wk, Wv)
print(out.shape)

tokens = ["The", "cat", "sat", "on", "the", "mat"]
plt.imshow(weights, cmap="viridis"); plt.xticks(range(seq), tokens); plt.yticks(range(seq), tokens)
plt.xlabel("key (attended to)"); plt.ylabel("query"); plt.title("causal attention weights (random weights)"); plt.colorbar(); plt.show()

The upper triangle is zero: token 3 can only attend to tokens 0–3. That's the **causal mask**. During training, the model
predicts every next token in a sequence in parallel, and the mask stops it from cheating by peeking at the answer.

## 4. Check against PyTorch

In [ ]:
Xt, Wqt, Wkt, Wvt = (torch.tensor(a, dtype=torch.float32) for a in (X, Wq, Wk, Wv))
ref = nn.functional.scaled_dot_product_attention(Xt @ Wqt, Xt @ Wkt, Xt @ Wvt, is_causal=True)
print("max difference vs torch:", (ref - torch.tensor(out, dtype=torch.float32)).abs().max().item())
assert torch.allclose(ref, torch.tensor(out, dtype=torch.float32), atol=1e-5)

`scaled_dot_product_attention` dispatches to fused kernels (FlashAttention-style) that never materialise the full
seq × seq matrix — crucial for long contexts.

## 5. Multi-head attention
Run $h$ smaller attentions in parallel (each with $d_k = d/h$), concatenate, project. Different heads learn different
relations (previous token, matching brackets, subject–verb…).

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_k = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)  # all three projections in one matmul
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        # (B, T, C) -> (B, h, T, d_k): heads become a batch dimension
        q, k, v = (t.view(B, T, self.h, self.d_k).transpose(1, 2) for t in (q, k, v))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
        weights = scores.masked_fill(mask, float("-inf")).softmax(dim=-1)
        out = (weights @ v).transpose(1, 2).reshape(B, T, C)  # concatenate heads
        return self.proj(out)


mha = MultiHeadAttention(d_model=64, n_heads=4)
x = torch.randn(2, 10, 64)
print(mha(x).shape)

## 6. Cost: attention is O(T²)
The score matrix has T² entries per head per layer. Doubling the context quadruples attention compute and memory —
why long context is expensive, and why tricks like FlashAttention, sliding windows, and KV caching (chapter 13) exist.

In [ ]:
for T in [1_000, 8_000, 128_000, 1_000_000]:
    print(f"context {T:>9,}: {T * T * 2 / 1e9:>10,.1f} GB for ONE fp16 score matrix (one head, one layer)")

## Exercises
1. Make the toy attention ignore positions: shuffle the input rows of `X` (without the mask). Show the outputs are shuffled the same way (permutation equivariance).
2. Implement RoPE: rotate pairs of dimensions of q and k by angle `pos * θ_i`. Verify that `q_m · k_n` depends only on `m - n`.
3. Write the multi-head scores with `torch.einsum("bhqd,bhkd->bhqk", q, k)` and check it matches.

In [ ]:
# your code here